<a href="https://colab.research.google.com/github/AaronYounger/Machine-Learning/blob/main/Reinforcement_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Q-Learning:

Given the Environment (10x10 boxes) where a Robotic Agent should move from a given position to the Goal Position (green box) using the shortest possible route.



Write the Python code to set up the environment:

- Build the 10x10 array

- Rewards in Black boxes: -100

- Rewards in White boxes (pathway): -1

- Reward for Goal box: 100



Then build the Q-value matrix by training the model, and find the shortest path from any given position to the goal position.

In [1]:
# Import Packages
import numpy as np

In [2]:
environment_rows = 10
environment_columns = 10
q_values = np.zeros((environment_rows, environment_columns, 4))
#print("Q-Values:", q_values)
#0 = up, 1 = right, 2 = down, 3=left
actions = ['up', 'right', 'down', 'left']

In [4]:
# Set reward for black boxes
rewards = np.full((environment_rows, environment_columns), -100)
rewards[0,2] = 100 # Reward for Green Box
print("Rewards:", rewards)

Rewards: [[-100 -100  100 -100 -100 -100 -100 -100 -100 -100]
 [-100 -100 -100 -100 -100 -100 -100 -100 -100 -100]
 [-100 -100 -100 -100 -100 -100 -100 -100 -100 -100]
 [-100 -100 -100 -100 -100 -100 -100 -100 -100 -100]
 [-100 -100 -100 -100 -100 -100 -100 -100 -100 -100]
 [-100 -100 -100 -100 -100 -100 -100 -100 -100 -100]
 [-100 -100 -100 -100 -100 -100 -100 -100 -100 -100]
 [-100 -100 -100 -100 -100 -100 -100 -100 -100 -100]
 [-100 -100 -100 -100 -100 -100 -100 -100 -100 -100]
 [-100 -100 -100 -100 -100 -100 -100 -100 -100 -100]]


In [5]:
# Set up environment
aisles = {}
aisles[1] = [i for i in range(1,9)]
aisles[2] = [1, 6, 9]
aisles[3] = [i for i in range(1, 7)]
aisles[4] = [1, 3, 6, 7, 8, 9]
aisles[5] = [1, 3, 4, 5, 6]
aisles[6] = [1, 3, 5, 6]
aisles[7] = [3, 6, 8]
aisles[8] = [i for i in range(0, 9)]
aisles

{1: [1, 2, 3, 4, 5, 6, 7, 8],
 2: [1, 6, 9],
 3: [1, 2, 3, 4, 5, 6],
 4: [1, 3, 6, 7, 8, 9],
 5: [1, 3, 4, 5, 6],
 6: [1, 3, 5, 6],
 7: [3, 6, 8],
 8: [0, 1, 2, 3, 4, 5, 6, 7, 8]}

In [8]:
# Set rewards for white squares
for r in range (1,9):
  for c in aisles[r]:
    rewards[r, c] = -1

rewards

array([[-100, -100,  100, -100, -100, -100, -100, -100, -100, -100],
       [-100,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1, -100],
       [-100,   -1, -100, -100, -100, -100,   -1, -100, -100,   -1],
       [-100,   -1,   -1,   -1,   -1,   -1,   -1, -100, -100, -100],
       [-100,   -1, -100,   -1, -100, -100,   -1,   -1,   -1,   -1],
       [-100,   -1, -100,   -1,   -1,   -1,   -1, -100, -100, -100],
       [-100,   -1, -100,   -1, -100,   -1,   -1, -100, -100, -100],
       [-100, -100, -100,   -1, -100, -100,   -1, -100,   -1, -100],
       [  -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1, -100],
       [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100]])

In [9]:
# Function to determine whether a state is terminal, If the reward is -1 it is not terminal otherwise terminal
def is_terminal_state(current_r, current_c):
    if rewards[current_r, current_c] == -1:
        return False
    else:
        return True

# Pick a random valid starting location
def get_starting_location():
    current_r = np.random.randint(environment_rows)
    current_c = np.random.randint(environment_columns)
    while is_terminal_state(current_r, current_c):
        current_r = np.random.randint(environment_rows)
        current_c = np.random.randint(environment_columns)
    return current_r, current_c

# Choose an action usign epsilon logic
def get_next_action(current_row_index, current_column_index, epsilon):
    if np.random.random() < epsilon:
        return np.argmax(q_values[current_row_index, current_column_index])
    else:
        return np.random.randint(4)

# Function to move the agent to the next location
def get_next_location(current_row_index, current_column_index, action_index):
    new_row_index = current_row_index
    new_column_index = current_column_index

    if actions[action_index] == 'up' and current_row_index > 0:
        new_row_index -= 1
    elif actions[action_index] == 'right' and current_column_index < environment_columns - 1:
        new_column_index += 1
    elif actions[action_index] == 'down' and current_row_index < environment_rows - 1:
        new_row_index += 1
    elif actions[action_index] == 'left' and current_column_index > 0:
        new_column_index -= 1

    return new_row_index, new_column_index

# Follow the learned best path after training
def get_shortest_path(start_row_index, start_column_index):
    if is_terminal_state(start_row_index, start_column_index):
        return []
    else:  # if this is a 'legal' starting location
        current_row_index, current_column_index = start_row_index, start_column_index
        shortest_path = []
        shortest_path.append([current_row_index, current_column_index])

        while not is_terminal_state(current_row_index, current_column_index):
            action_index = get_next_action(current_row_index, current_column_index, 1.)
            current_row_index, current_column_index = get_next_location(
                current_row_index, current_column_index, action_index
            )
            shortest_path.append([current_row_index, current_column_index])

        return shortest_path


# Train Q-values over 1000 episodes
epsilon = 0.9
discount_factor = 0.9
learning_rate = 0.9

for episode in range(1000):
    row_index, column_index = get_starting_location()

    while not is_terminal_state(row_index, column_index):
        action_index = get_next_action(row_index, column_index, epsilon)
        old_row_index, old_column_index = row_index, column_index
        row_index, column_index = get_next_location(row_index, column_index, action_index)
        reward = rewards[row_index, column_index]
        old_q_value = q_values[old_row_index, old_column_index, action_index]
        temporal_difference = reward + (
            discount_factor * np.max(q_values[row_index, column_index])
        ) - old_q_value
        new_q_value = old_q_value + (learning_rate * temporal_difference)
        q_values[old_row_index, old_column_index, action_index] = new_q_value

print('Training complete!')



Training complete!
[[4, 3], [3, 3], [3, 2], [3, 1], [2, 1], [1, 1], [1, 2], [0, 2]]
[]
[[8, 2], [8, 3], [7, 3], [6, 3], [5, 3], [4, 3], [3, 3], [3, 2], [3, 1], [2, 1], [1, 1], [1, 2], [0, 2]]


In [21]:
# Print Q-Values
print(q_values)

[[[   0.            0.            0.            0.        ]
  [   0.            0.            0.            0.        ]
  [   0.            0.            0.            0.        ]
  [   0.            0.            0.            0.        ]
  [   0.            0.            0.            0.        ]
  [   0.            0.            0.            0.        ]
  [   0.            0.            0.            0.        ]
  [   0.            0.            0.            0.        ]
  [   0.            0.            0.            0.        ]
  [   0.            0.            0.            0.        ]]

 [[   0.            0.            0.            0.        ]
  [ -99.99999      89.           70.18999288  -99.999999  ]
  [ 100.           79.1        -100.           79.1       ]
  [ -99.99         70.19       -100.           89.        ]
  [ -99.9999999    62.171       -99.9999999    79.1       ]
  [-100.           54.95389994 -100.           70.19      ]
  [-100.           48.45851      48.45

In [20]:
# Find first Shortest path (Goal Box = 1, 2)
print(get_shortest_path(8, 7))

[[8, 7], [8, 6], [7, 6], [6, 6], [5, 6], [4, 6], [3, 6], [2, 6], [1, 6], [1, 5], [1, 4], [1, 3], [1, 2], [0, 2]]


In [19]:
# Find second Shortest path (Goal Box = 1, 2)
print(get_shortest_path(1, 8))

[[1, 8], [1, 7], [1, 6], [1, 5], [1, 4], [1, 3], [1, 2], [0, 2]]


In [18]:
# Find third Shortest path (Goal Box = 1, 2)
print(get_shortest_path(4, 9))

[[4, 9], [4, 8], [4, 7], [4, 6], [3, 6], [2, 6], [1, 6], [1, 5], [1, 4], [1, 3], [1, 2], [0, 2]]
